# Setup GTEx data
## Setup log normalized TPM data of lncRNA-TF pairs
### Author: Martin Loza
### Date: 25/12/18

Now, let's select expression data of selected gene pairs

In [1]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
})

# Local variables 
seed = 777
date = "251218"

# Define colors for strand plots
red = "#E41A1C"
blue = "#090a0bff"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
text_size = 18
width = 18.6
dot_size = 4
line_size = 1.5
dpi = 300

data_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/normalized/"
out_dir = "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/selected_gene_pairs/"

# Local Functions


### Load and setup the data

In [2]:
# load the merged normalized data
data_normalized <- read.table(paste0(data_dir, "log_normalized_tpm_251218.tsv"), 
                        header = TRUE, sep = "\t", stringsAsFactors = FALSE)
# Remove the Description column for easier handling
data_normalized <- data_normalized %>%
    select(-Description)
head(data_normalized)[1:5]

,Name,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972,0.000000,0.000000,0.00000000,0.000000
2,ENSG00000227232,1.609016,1.429869,1.30103055,1.623499
3,ENSG00000278267,0.000000,0.000000,0.00000000,0.000000
4,ENSG00000243485,0.000000,0.000000,0.00000000,0.000000
5,ENSG00000237613,0.000000,0.000000,0.00000000,0.000000
6,ENSG00000268020,0.000000,0.000000,0.03514415,0.000000


In [3]:
# Extract sample names from column names (remove "log_TPM_" prefix)
sample_names <- colnames(data_normalized)[-c(1)]  # Exclude gene Name column

cat("Number of samples:", length(sample_names), "\n")
head(sample_names)

Number of samples: 68 


[1] "Adipose_Subcutaneous"     "Adipose_Visceral_Omentum"
[3] "Adrenal_Gland"            "Artery_Aorta"            
[5] "Artery_Coronary"          "Artery_Tibial"

Load the lncRNA-TF gene pairs

In [4]:
gene_pairs_file <- "/Users/martin/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
# Load the unique gene pairs data
gene_pairs <- read.table(gene_pairs_file, sep = "\t", header = TRUE, comment.char = "", fill = TRUE, row.names = NULL)
dim(gene_pairs)
head(gene_pairs,2)

[1] 1978   18

,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,strand_distance,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id,gene_pair_id,gene_name_pair_id
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>
1,11,ENST00000381363,2140644,IGF2-AS,1,lncRNA,ENST00000643349,unnamed,2149603,8959,8959,ZBTB,TRUE,8959,ENSG00000099869,ENSG00000284779,ENSG00000099869_ENSG00000284779,IGF2-AS_unnamed
2,11,ENST00000833483,61756482,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-1093,1093,NDT80_PhoG,TRUE,1093,ENSG00000124915,ENSG00000124920,ENSG00000124915_ENSG00000124920,MYRF-AS1_MYRF


In [5]:
# get unique lncRNA and TF gene IDs
unique_gene_pairs_ids <- unique(c(gene_pairs$ncrna_gene_id, gene_pairs$pcg_gene_id))
# get unique gene_ids in the GETx data. 
unique_gtex_gene_ids <- unique(data_normalized$Name)

cat("Number of unique lncRNA and TF gene IDs: ", length(unique_gene_pairs_ids), "\n")
cat("Number of unique lncRNA and TF gene IDs in gene pairs: ", sum(unique_gene_pairs_ids %in% unique_gtex_gene_ids), "\n")

Number of unique lncRNA and TF gene IDs:  3074 


Number of unique lncRNA and TF gene IDs in gene pairs:  2057 


We don't have expression data of all genes...

 Let's then select only expression data of related gene pairs

In [6]:
# Select only expression data of genes present in the gene pairs
data_normalized_selected <- data_normalized %>%
    filter(Name %in% unique_gene_pairs_ids)
cat("Number of genes in merged normalized data after selection: ", nrow(data_normalized_selected), "\n")

Number of genes in merged normalized data after selection:  2058 


In [7]:
head(data_normalized_selected)[1:5]

,Name,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000272438,0.06808566,0.1234452,0.27456568,0.2352768
2,ENSG00000223764,0.46315775,0.9997234,1.60198420,1.9059241
3,ENSG00000187634,0.39083866,1.3746670,1.58750105,1.9401048
4,ENSG00000272512,1.82029833,1.5857946,0.49259098,2.2775975
5,ENSG00000188290,3.16147126,3.2602481,1.38873139,3.8918407
6,ENSG00000197921,0.23182734,0.3240786,0.02956657,0.3727732


In [12]:
# Save the merged data
out_file <- file.path(out_dir, paste0("log_normalized_tpm_selected_lncRNA_TF_genes_", date, ".tsv"))
write.table(data_normalized_selected, file = out_file, sep = "\t", quote = FALSE, row.names = FALSE)